# D84 敏感性重跑(lr 日程对齐版; 取代 d83 的 50 轮扫描结果)

**背景**: d83 的 50 轮 run 因 `--iters` 同时决定 lr 退火跨度而背 handicap(空白重跑 138.6k vs 中心 123.6k),
对 100 轮中心画图会产生 ~+12% 假性劣化。本版全部 `--iters 100`(与 d73_AB 完全同日程), byR 全程选,
中心 = full method **123,035.5**(不另跑)。批量组等总环境步 819,200、退火随各自长度。

**不重跑**: promote-delay-k(已证明不生效——三个 k 的 run 逐位相同, 面板为可证明的平线)。
**新 tag 一律 `_i100` 后缀**, 不覆盖 d83 的任何文件。C 组顺带修复 D76 两个消融 run 的同一问题。
全部 cell 重跑安全(`--resume` 幂等)。**耗时校准(按 d73_AB 实测 4.94h/100 轮)**: 单 run 约 2–5h,
16 run 并行(每 run 4 线程,共 64≤72 核)一晚可完。**尽早发射**; 万一早上个别 run 未完,
json 是边跑边写的——照常拷回,byR 取已有 evals 仍有效(同日程,只是选择点少),画图脚本自动容忍。
批量组 eval 间隔按等环境步对齐(2048→ee20 / 4096→ee10 / 16384→ee2),与中心同为 20–25 次评估。


In [ ]:
import sys, os
venv_site = '/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/lib/python3.11/site-packages'
if os.path.isdir(venv_site) and venv_site not in sys.path:
    sys.path.insert(0, venv_site)
%cd /rds/homes/h/hxn886/ondemand/final_project
PROJ='/rds/homes/h/hxn886/ondemand/final_project'
PY='/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/bin/python'
PREFIX=(f'cd {PROJ} && env PYTHONUNBUFFERED=1 PYTHONPATH=. OMP_NUM_THREADS=4 MKL_NUM_THREADS=4 OPENBLAS_NUM_THREADS=4 '
        f'PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True MPLBACKEND=Agg {PY}')
print('OK')


## 预检


In [ ]:
import os
fails = []
def chk(cond, msg):
    print(('PASS ' if cond else 'FAIL ') + msg)
    if not cond: fails.append(msg)
env = open('derbysim/sim/env_dynamic.py').read(); s13 = open('scripts/13_train_ppo.py').read()
chk('control_classes' in env, 'env_dynamic 新版 (D67)')
chk('--fail-penalty' in s13 and '--bc-anchor-beta' in s13, 'scripts/13 旗标齐全')
chk(os.path.exists('artifacts/reports/ppo_push_AB_s0_best.pt'), 'init ckpt 在位')
if fails: raise SystemExit('预检失败! 同步 derbysim/ + scripts/13 后重跑本 cell')
print('预检通过')


## A. β 双锚(5 run, 各约 1.1h)


In [ ]:
import subprocess, os
os.makedirs('rl_logs', exist_ok=True)
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_br001_i100 --kl-beta 0.01 --bc-anchor-beta 0.01 --lr 5e-5 --steps-per-iter 8192 --iters 100 --fail-penalty 50 --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --promote-delay-k 1.0 --kl-anchor schedule --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 5 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_br001_i100.log 2>&1 &'])
print('launched sens_br001_i100')
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_br01_i100 --kl-beta 0.1 --bc-anchor-beta 0.01 --lr 5e-5 --steps-per-iter 8192 --iters 100 --fail-penalty 50 --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --promote-delay-k 1.0 --kl-anchor schedule --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 5 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_br01_i100.log 2>&1 &'])
print('launched sens_br01_i100')
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_bt0_i100 --kl-beta 0.03 --bc-anchor-beta 0 --lr 5e-5 --steps-per-iter 8192 --iters 100 --fail-penalty 50 --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --promote-delay-k 1.0 --kl-anchor schedule --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 5 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_bt0_i100.log 2>&1 &'])
print('launched sens_bt0_i100')
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_bt0005_i100 --kl-beta 0.03 --bc-anchor-beta 0.005 --lr 5e-5 --steps-per-iter 8192 --iters 100 --fail-penalty 50 --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --promote-delay-k 1.0 --kl-anchor schedule --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 5 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_bt0005_i100.log 2>&1 &'])
print('launched sens_bt0005_i100')
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_bt002_i100 --kl-beta 0.03 --bc-anchor-beta 0.02 --lr 5e-5 --steps-per-iter 8192 --iters 100 --fail-penalty 50 --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --promote-delay-k 1.0 --kl-anchor schedule --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 5 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_bt002_i100.log 2>&1 &'])
print('launched sens_bt002_i100')


## B. lr / 搁浅罚(6 run, 各约 1.1h)


In [ ]:
import subprocess, os
os.makedirs('rl_logs', exist_ok=True)
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_lr2e5_i100 --kl-beta 0.03 --bc-anchor-beta 0.01 --lr 2e-5 --steps-per-iter 8192 --iters 100 --fail-penalty 50 --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --promote-delay-k 1.0 --kl-anchor schedule --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 5 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_lr2e5_i100.log 2>&1 &'])
print('launched sens_lr2e5_i100')
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_lr1e4_i100 --kl-beta 0.03 --bc-anchor-beta 0.01 --lr 1e-4 --steps-per-iter 8192 --iters 100 --fail-penalty 50 --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --promote-delay-k 1.0 --kl-anchor schedule --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 5 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_lr1e4_i100.log 2>&1 &'])
print('launched sens_lr1e4_i100')
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_lr2e4_i100 --kl-beta 0.03 --bc-anchor-beta 0.01 --lr 2e-4 --steps-per-iter 8192 --iters 100 --fail-penalty 50 --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --promote-delay-k 1.0 --kl-anchor schedule --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 5 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_lr2e4_i100.log 2>&1 &'])
print('launched sens_lr2e4_i100')
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_fp25_i100 --kl-beta 0.03 --bc-anchor-beta 0.01 --lr 5e-5 --steps-per-iter 8192 --iters 100 --fail-penalty 25 --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --promote-delay-k 1.0 --kl-anchor schedule --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 5 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_fp25_i100.log 2>&1 &'])
print('launched sens_fp25_i100')
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_fp100_i100 --kl-beta 0.03 --bc-anchor-beta 0.01 --lr 5e-5 --steps-per-iter 8192 --iters 100 --fail-penalty 100 --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --promote-delay-k 1.0 --kl-anchor schedule --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 5 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_fp100_i100.log 2>&1 &'])
print('launched sens_fp100_i100')
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_fp200_i100 --kl-beta 0.03 --bc-anchor-beta 0.01 --lr 5e-5 --steps-per-iter 8192 --iters 100 --fail-penalty 200 --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --promote-delay-k 1.0 --kl-anchor schedule --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 5 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_fp200_i100.log 2>&1 &'])
print('launched sens_fp200_i100')


## C. 消融修复重跑(2 run;修 D76 的 50 轮日程 handicap)


In [ ]:
import subprocess, os
os.makedirs('rl_logs', exist_ok=True)
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag abl_noroute_i100 --kl-beta 0 --bc-anchor-beta 0.01 --lr 5e-5 --steps-per-iter 8192 --iters 100 --fail-penalty 50 --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --promote-delay-k 1.0 --kl-anchor none --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 5 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/abl_noroute_i100.log 2>&1 &'])
print('launched abl_noroute_i100')
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag abl_nocur_i100 --kl-beta 0.03 --bc-anchor-beta 0.01 --lr 5e-5 --steps-per-iter 8192 --iters 100 --fail-penalty 50 --kl-anchor schedule --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 5 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/abl_nocur_i100.log 2>&1 &'])
print('launched abl_nocur_i100')


## D. 批量(等总环境步 819,200, 退火随各自长度; 2048 约 4h)


In [ ]:
import subprocess, os
os.makedirs('rl_logs', exist_ok=True)
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_sp2048_i100 --kl-beta 0.03 --bc-anchor-beta 0.01 --lr 5e-5 --steps-per-iter 2048 --iters 400 --fail-penalty 50  --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --promote-delay-k 1.0 --kl-anchor schedule --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 20 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_sp2048_i100.log 2>&1 &'])
print('launched sens_sp2048_i100')
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_sp4096_i100 --kl-beta 0.03 --bc-anchor-beta 0.01 --lr 5e-5 --steps-per-iter 4096 --iters 200 --fail-penalty 50  --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --promote-delay-k 1.0 --kl-anchor schedule --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 10 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_sp4096_i100.log 2>&1 &'])
print('launched sens_sp4096_i100')
cmd = f'{PREFIX} scripts/13_train_ppo.py --run-tag sens_sp16384_i100 --kl-beta 0.03 --bc-anchor-beta 0.01 --lr 5e-5 --steps-per-iter 16384 --iters 50 --fail-penalty 50  --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 1.0 --promote-delay-k 1.0 --kl-anchor schedule --feature-version 2 --seed 0 --init-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --anchor-ckpt artifacts/reports/ppo_push_AB_s0_best.pt --encoder gnn --control-classes 12 --reserve-window wide --train-eval-days 6 --eval-every 2 --early-stop-patience 8 --best-rescue --resume'
subprocess.Popen(['bash','-c', f'nohup bash -c "{cmd}" > rl_logs/sens_sp16384_i100.log 2>&1 &'])
print('launched sens_sp16384_i100')


## 监控


In [ ]:
import glob, subprocess
for lg in sorted(glob.glob('rl_logs/*_i100.log')):
    out = subprocess.run(['tail','-n','2',lg], capture_output=True, text=True).stdout.strip()
    print('====', lg); print(out)


## 汇总(全部跑完后)


In [ ]:
import json, glob
def by_reward(f):
    evs = json.load(open(f))['evals']
    b = max(evs, key=lambda e: e['eval_episode_reward'])
    return b['iter'], b['eval_weighted_secondary_delay_s'], b['eval_completion_rate']
print(f"{'run':<20s}{'bestIt':>7s}{'delay':>13s}{'comp':>7s}")
print(f"{'CENTER(full 123.0k)':<20s}{74:>7d}{123035.5:>13,.1f}{1.0:>7.3f}")
for f in sorted(glob.glob('artifacts/reports/*_i100.json')):
    it, d, c = by_reward(f)
    print(f"{f.split('/')[-1][:-5]:<20s}{it:>7d}{d:>13,.1f}{c:>7.3f}")


## 完成后拷回本地

拷回 `artifacts/reports/*_i100.json`(16 个)到本地同路径, 然后本地跑:
`python scripts/45_sensitivity_grid.py`(F17 六联大图)、`43`(F15)、`44`(F16)、
`python scripts/42_ablation_v2.py`(消融表自动改用修复后的 _i100 两行)。
